# CS-272 Artificial Intelligence: Lab 6
### Informed Search - Greedy Best-First Search and Heuristics

**Name:** Arqam Waheed
**Enrollment No.:** 553514
**Class:** BSDS 2K25
**Lab:** 06

## Part A - Upload and Run the Lab 5 Project

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os, zipfile

zip_name = next(name for name in uploaded if name.endswith(".zip"))
extract_root = "/content/lab06_pacman"

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_root)

project_dir = None
for root, dirs, filenames in os.walk(extract_root):
    if "pacman.py" in filenames and "search.py" in filenames:
        project_dir = root
        break

assert project_dir is not None, "Pac-Man project folder was not found."
os.chdir(project_dir)
print("Project folder:", project_dir)

In [ ]:
!pwd
!ls

**Checkpoint A** - sanity check before editing any file.

In [ ]:
!python pacman.py -l tinyMaze -p SearchAgent -a fn=tinyMazeSearch -t --frameTime 0

[SearchAgent] using function tinyMazeSearch
[SearchAgent] using problem type PositionSearchProblem
%%%%%%%
%    <%
% %%% %
%  %  %
%%   %%
%. %%%%
%%%%%%%
Score: 0

Path found with total cost of 8 in 0.0 seconds
Search nodes expanded: 0
%%%%%%%
%     %
% %%%^%
%  %  %
%%   %%
%. %%%%
%%%%%%%
Score: -1

%%%%%%%
%     %
% %%% %
%  % ^%
%%   %%
%. %%%%
%%%%%%%
Score: -2

%%%%%%%
%     %
% %%% %
%  %> %
%%   %%
%. %%%%
%%%%%%%
Score: -3

%%%%%%%
%     %
% %%% %
%  %  %
%%  ^%%
%. %%%%
%%%%%%%
Score: -4

%%%%%%%
%     %
% %%% %
%  %  %
%% > %%
%. %%%%
%%%%%%%
Score: -5

%%%%%%%
%     %
% %%% %
%  %  %
%%>  %%
%. %%%%
%%%%%%%
Score: -6

%%%%%%%
%     %
% %%% %
%  %  %
%%   %%
%.^%%%%
%%%%%%%
Score: -7

%%%%%%%
%     %
% %%% %
%  %  %
%%   %%
%> %%%%
%%%%%%%
Score: 502

%%%%%%%
%     %
% %%% %
%  %  %
%%   %%
%> %%%%
%%%%%%%
Score: 502

Pacman emerges victorious! Score: 502
Average Score: 502.0
Scores:        502.0
Win Rate:      1/1 (1.00)
Record:        Win


## Part B - Priority Queue and Heuristic Values

Smallest priority is removed first.

In [ ]:
import util

pq = util.PriorityQueue()

pq.push("A", 7)
pq.push("B", 2)
pq.push("C", 5)

print("First removed:", pq.pop())
print("Second removed:", pq.pop())

First removed: B
Second removed: C


For Greedy Best-First Search the priority is the heuristic value:

```
priority = h(n)

smaller h(n)  -> estimated closer to goal
larger h(n)   -> estimated farther from goal
```

Manhattan: `h(n) = |x1 - x2| + |y1 - y2|`

In [ ]:
current = (2, 3)
goal = (5, 7)

manhattan = abs(current[0] - goal[0]) + abs(current[1] - goal[1])
print("Manhattan h(n):", manhattan)

Manhattan h(n): 7


Euclidean: `h(n) = sqrt((x1 - x2)^2 + (y1 - y2)^2)`

In [ ]:
current = (2, 3)
goal = (5, 7)

euclidean = (
    (current[0] - goal[0]) ** 2
    + (current[1] - goal[1]) ** 2
) ** 0.5

print("Euclidean h(n):", euclidean)

Euclidean h(n): 5.0


| Feature | Manhattan | Euclidean |
|---|---|---|
| Meaning | horizontal + vertical distance | straight-line distance |
| Natural fit | four-direction grid movement | continuous / diagonal movement |
| Value from (2,3) to (5,7) | 7 | 5.0 |

## Part C - Existing Heuristic Interface

In [ ]:
import searchAgents
import inspect

print(inspect.getsource(searchAgents.manhattanHeuristic))

def manhattanHeuristic(position, problem, info={}):
    "The Manhattan distance heuristic for a PositionSearchProblem"
    xy1 = position
    xy2 = problem.goal
    return abs(xy1[0] - xy2[0]) + abs(xy1[1] - xy2[1])



## Activity 1 - Student Euclidean Heuristic (02 Marks)

Added in `searchAgents.py` below the existing heuristic functions.

In [ ]:
def studentEuclideanHeuristic(position, problem, info={}):
    "Student Euclidean distance heuristic for a PositionSearchProblem"
    x1, y1 = position
    x2, y2 = problem.goal

    dx = x1 - x2
    dy = y1 - y2

    return (dx ** 2 + dy ** 2) ** 0.5

In [ ]:
!python -m py_compile searchAgents.py

## Activity 2 - Greedy Best-First Search (04 Marks)

Added in `search.py` below `nullHeuristic()`.

```python
def greedyBestFirstSearch(problem, heuristic=nullHeuristic):
    """Search the node with the smallest heuristic value h(n) first."""
    frontier = util.PriorityQueue()
    explored = set()

    start_state = problem.getStartState()
    start_item = (start_state, [], 0)

    frontier.push(start_item, heuristic(start_state, problem))

    while not frontier.isEmpty():

        state, actions, path_cost = frontier.pop()

        if state in explored:
            continue

        if problem.isGoalState(state):
            return actions

        explored.add(state)

        for successor, action, step_cost in problem.getSuccessors(state):

            if successor not in explored:

                new_actions = actions + [action]
                new_cost = path_cost + step_cost

                priority = heuristic(successor, problem)

                frontier.push(
                    (successor, new_actions, new_cost),
                    priority
                )

    return []
```

Registered at the bottom of `search.py`:

```python
greedy = greedyBestFirstSearch
```

In [ ]:
!python -m py_compile search.py
!python -m py_compile searchAgents.py

## Part E - Run Greedy Search

tinyMaze with Manhattan. Legend: `%` = wall, `.` = goal, `< > ^ v` = Pac-Man.

In [ ]:
!python pacman.py -l tinyMaze -p SearchAgent -a fn=greedy,heuristic=manhattanHeuristic -t --frameTime 0

[SearchAgent] using function greedy and heuristic manhattanHeuristic
[SearchAgent] using problem type PositionSearchProblem
%%%%%%%
%    <%
% %%% %
%  %  %
%%   %%
%. %%%%
%%%%%%%
Score: 0

Path found with total cost of 8 in 0.0 seconds
Search nodes expanded: 8
%%%%%%%
%     %
% %%%^%
%  %  %
%%   %%
%. %%%%
%%%%%%%
Score: -1

%%%%%%%
%     %
% %%% %
%  % ^%
%%   %%
%. %%%%
%%%%%%%
Score: -2

%%%%%%%
%     %
% %%% %
%  %> %
%%   %%
%. %%%%
%%%%%%%
Score: -3

%%%%%%%
%     %
% %%% %
%  %  %
%%  ^%%
%. %%%%
%%%%%%%
Score: -4

%%%%%%%
%     %
% %%% %
%  %  %
%% > %%
%. %%%%
%%%%%%%
Score: -5

%%%%%%%
%     %
% %%% %
%  %  %
%%>  %%
%. %%%%
%%%%%%%
Score: -6

%%%%%%%
%     %
% %%% %
%  %  %
%%   %%
%.^%%%%
%%%%%%%
Score: -7

%%%%%%%
%     %
% %%% %
%  %  %
%%   %%
%> %%%%
%%%%%%%
Score: 502

%%%%%%%
%     %
% %%% %
%  %  %
%%   %%
%> %%%%
%%%%%%%
Score: 502

Pacman emerges victorious! Score: 502
Average Score: 502.0
Scores:        502.0
Win Rate:      1/1 (1.00)
Record:        Win


mediumMaze with Manhattan.

In [ ]:
!python pacman.py -l mediumMaze -p SearchAgent -a fn=greedy,heuristic=manhattanHeuristic -q

[SearchAgent] using function greedy and heuristic manhattanHeuristic
[SearchAgent] using problem type PositionSearchProblem
Path found with total cost of 74 in 0.0 seconds
Search nodes expanded: 78
Pacman emerges victorious! Score: 436
Average Score: 436.0
Scores:        436.0
Win Rate:      1/1 (1.00)
Record:        Win


mediumMaze with the student Euclidean heuristic.

In [ ]:
!python pacman.py -l mediumMaze -p SearchAgent -a fn=greedy,heuristic=studentEuclideanHeuristic -q

[SearchAgent] using function greedy and heuristic studentEuclideanHeuristic
[SearchAgent] using problem type PositionSearchProblem
Path found with total cost of 152 in 0.0 seconds
Search nodes expanded: 159
Pacman emerges victorious! Score: 358
Average Score: 358.0
Scores:        358.0
Win Rate:      1/1 (1.00)
Record:        Win


## Activity 3 - Compare Heuristics and Search Behavior (04 Marks)

Greedy Best-First Search on mediumMaze:

| Heuristic | Nodes Expanded | Solution Length / Cost | Reached Goal? | Observation |
|---|---|---|---|---|
| Manhattan | 78 | 74 | Yes | |
| Student Euclidean | 159 | 152 | Yes | |

Greedy vs BFS on mediumMaze (BFS numbers recorded in Lab 5):

| Algorithm | Information Used | Nodes Expanded | Solution Length / Cost | Optimal Here? |
|---|---|---|---|---|
| BFS | No heuristic | 269 | 68 | |
| Greedy (Manhattan) | h(n) | 78 | 74 | |

### Analysis Questions

1. What information does h(n) provide that DFS/BFS did not use?
2. Why does Greedy choose the state with the smallest h(n)?
3. Which heuristic expanded fewer nodes in your mediumMaze run? Support the answer with your table.
4. Did the heuristic that expanded fewer nodes also produce the shorter path? Explain using your results.
5. Why is Greedy Best-First Search not guaranteed to be optimal even when h(n) appears to point toward the goal?
6. State one practical advantage and one limitation of Greedy search.
7. In one sentence, explain why A* will need more than h(n) alone.

## Extension Task - Uniform Cost Search (Optional / Not Graded)

```
UCS priority    = g(n) = actual path cost from Start to n
Greedy priority = h(n) = estimated remaining cost from n to Goal
```

The `uniformCostSearch(problem)` stub in `search.py` was completed.

```python
def uniformCostSearch(problem):
    """Search the node of least total cost first."""
    frontier = util.PriorityQueue()
    explored = set()

    start_state = problem.getStartState()
    start_item = (start_state, [], 0)

    frontier.push(start_item, 0)

    while not frontier.isEmpty():

        state, actions, path_cost = frontier.pop()

        if state in explored:
            continue

        if problem.isGoalState(state):
            return actions

        explored.add(state)

        for successor, action, step_cost in problem.getSuccessors(state):

            if successor not in explored:
                new_actions = actions + [action]
                new_cost = path_cost + step_cost

                frontier.push(
                    (successor, new_actions, new_cost),
                    new_cost
                )

    return []
```

The alias `ucs = uniformCostSearch` was already present.

In [ ]:
!python pacman.py -l mediumMaze -p SearchAgent -a fn=ucs -q

[SearchAgent] using function ucs
[SearchAgent] using problem type PositionSearchProblem
Path found with total cost of 68 in 0.0 seconds
Search nodes expanded: 269
Pacman emerges victorious! Score: 442
Average Score: 442.0
Scores:        442.0
Win Rate:      1/1 (1.00)
Record:        Win


| Algorithm | Priority Rule | Nodes Expanded | Path Cost | Solution Length | Optimal? |
|---|---|---|---|---|---|
| UCS | g(n) | 269 | 68 | 68 | Yes, for non-negative step costs |
| Greedy | h(n) | 78 | 74 | 74 | Not guaranteed |

### Short Analysis

1. UCS and Greedy both use a Priority Queue. What exactly is different about their priority values?
2. Which algorithm expanded fewer nodes in your mediumMaze experiment?
3. Did the algorithm that expanded fewer nodes also return the lower-cost path?
4. Why can UCS guarantee the least-cost solution with non-negative step costs while Greedy cannot?
5. Which algorithm uses information about the goal location?

## Download

In [ ]:
from google.colab import files

files.download("search.py")
files.download("searchAgents.py")

## Reflection

What changed when search became informed? Which heuristic would you choose for this Pac-Man
movement model, and why?